**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/yang-2025"], check=True)
    os.chdir("/content/encore/entries/yang-2025")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Yang et al. (2025) from dandiset 001212

**Paper.** Yang, Ni, Sun, Xing, Wang, Zhang, Zhu and Xu (2025), "Individualized Target Selection of
Closed-Loop Electrical Stimulation for the Treatment of Spontaneous Temporal Lobe Epilepsy", IEEE
Trans. Neural Syst. Rehabil. Eng. 33:4008-4016, doi:10.1109/tnsre.2025.3614867.

**Dataset.** DANDI:001212, version `0.241023.0931` (doi:10.48324/dandi.001212/0.241023.0931),
streamed from the archive. The paper cites no version of the dandiset. This is the only published
version, and it is the most recent; the draft has the same 56 assets and size, so no other version
could match the claims file's description better.

**Claims.** `claims/yang-2025.yaml`, written from the version of record and its supplement (Figures
S1 to S3, Table S1), checked by a second reader. The claims file decides what is tested; method
details it leaves open are taken from the paper, cited where used, and the readings it names are
all run.

**What the dandiset holds.** For each of the 7 rats, 8 NWB files (4 baseline and 4 stimulation
groups); each file holds one 600 s, 4-channel (DG, CA1, CA3, ANT) LFP epoch per seizure at 1 kHz.
There are no seizure times, no stimulation times, no interictal recordings and no recording
durations. Seizure onset and end and the stimulation trains are therefore found from the LFP here
(an assumption, stated below), and the claims that need interictal data or recording time are
blocked.

| Claim | Tested or blocked |
|---|---|
| tables1-network-stable-short-term | tested |
| fig2-r1-target-shifts | tested |
| figs2-target-assignment | tested |
| fig3a-target-reduces-each-rat | tested |
| fig3a-nontarget-can-lengthen | tested |
| fig3a-per-rat-site-differences | tested |
| fig3a-target-vs-nontarget-total | tested |
| fig3a-some-nontarget-shorten | tested |
| fig3b-same-site-target-vs-nontarget | tested |
| seizure-duration-proportion | blocked: data missing (no recording time) |
| fig3c-frequency-no-difference | blocked: data missing (no recording time) |
| fig4b-psd-target-bands | tested |
| fig4b-psd-other-bands-unchanged | tested |
| fig5b-iis-post-seizure | tested |
| fig5c-interictal-iis-no-difference | blocked: data missing (no interictal segments) |
| fig5d-iis-seizure-correlation | blocked: data missing (no interictal segments, no recording time) |
| fig6cd-interictal-hfo-no-difference | blocked: data missing (no interictal segments) |
| fig6e-ictal-ripples-no-difference | tested |
| fig6f-ictal-fast-ripples-reduced | tested |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import time
import itertools
import resource
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import yang

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)
CH = yang.CHANNELS
print("dandiset", yang.DANDISET, "version", yang.VERSION)

## Loading

Every epoch of every file is streamed once, and everything the claims need is computed from it as
it is read (`yang.epoch_features`): seizure onset and end, stimulation trains, cf_w at onset, PSD
ratios, interictal spike counts around the seizure and ictal HFO rates. Four files are read at a
time, each in its own process.

In [ ]:
assets = yang.list_assets()
t0 = time.time()
rows = yang.process_all(assets, workers=4)
print(f"{len(assets)} files, {len(rows)} epochs, {time.time() - t0:.0f} s; peak memory of a worker "
      f"{resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / 1e6:.2f} GB")
df = pd.DataFrame(rows)
df["ntrains"] = df.trains.apply(len)
for th in yang.SZ_THRESH_VARIANTS:
    df[f"dur_{th}"] = df[f"sz_{th}"].apply(lambda x: x[1] - x[0])
df["dur_notrainend"] = df["sz_notrainend"].apply(lambda x: x[1] - x[0])
df["dur"] = df[f"dur_{yang.SZ_THRESH}"]

## Data check

In [ ]:
ad = pd.DataFrame(assets)
print("files per rat and kind:")
print(ad.groupby(["rat", "kind"]).size().unstack())
print("\nepochs per rat and kind:")
print(df.groupby(["rat", "kind"]).size().unstack())
print("\nepoch shapes:", df.groupby(["n_samples", "n_channels"]).size().to_dict(),
      "| rate (Hz):", df.rate.unique(), "| starting_time:", df.start.unique(), "| unit:", df.unit.unique())
print("LFP range over all epochs and channels (microvolts): min %.0f, max %.0f; per-epoch SD median %.1f "
      "(range %.1f to %.1f)" % (np.min(np.vstack(df["min"])), np.max(np.vstack(df["max"])),
                                np.median(np.vstack(df["std"])), np.min(np.vstack(df["std"])),
                                np.max(np.vstack(df["std"]))))
amax = np.maximum(np.abs(np.vstack(df["min"])), np.abs(np.vstack(df["max"]))).max(1)
print(f"epochs reaching |LFP| = {amax.max():.0f} microvolts (the largest value, likely the amplifier's "
      f"limit): {int((amax >= amax.max() - 1e-6).sum())} of {len(df)}; samples at that value in those epochs: "
      f"{int(df.n_at_extreme[amax >= amax.max() - 1e-6].sum())}")
print("each series' description names its own file's group:",
      all(r.description.split(" at ")[1].split(" ")[0].replace("_", "-") ==
          ad.set_index("group").loc[r.group, "path"].split(f"ses-Rat{r.rat}-")[1].split("_ecephys")[0]
          for r in df.itertuples()))

### Duplicates

Each epoch's data, and each channel separately, is hashed (MD5 of the float64 values).

In [ ]:
dup = df.md5.value_counts()
print("epochs whose full data occur more than once:", int((dup > 1).sum()))
chash = pd.Series([h for hs in df.md5_ch for h in hs]).value_counts()
print("channel traces that occur more than once:", int((chash > 1).sum()))
# epochs dated the same day in two groups of one rat
dd = df.groupby(["rat", "date"]).group.nunique()
print("rat-days with epochs in two groups:", dd[dd > 1].index.tolist())

### Group order: file names, epoch dates and Figure S2

The file names number the groups 00 to 07 (baseline, stimulation, alternating). The epoch names
carry dates. Figure S2 gives the order of the stimulated sites.

In [ ]:
FIGS2_ORDER = {  # stimulated site of stimulation groups 1 to 4, Figure S2 (claims file)
    "R1": ["CA1", "CA3", "ANT", "DG"], "R2": ["DG", "ANT", "CA1", "CA3"],
    "R3": ["CA3", "ANT", "DG", "CA1"], "R4": ["CA1", "DG", "ANT", "CA3"],
    "R5": ["ANT", "DG", "CA1", "CA3"], "R6": ["CA1", "ANT", "DG", "CA3"],
    "R7": ["DG", "ANT", "CA1", "CA3"]}
FIGS2_TARGETS = {  # computed target(s) of baseline groups 0 to 3, Figure S2
    "R1": [{"ANT"}, {"ANT"}, {"CA3"}, {"DG"}], "R2": [{"DG", "CA1"}, {"CA1"}, {"CA1"}, {"ANT"}],
    "R3": [{"CA3"}, {"CA3"}, {"ANT"}, {"CA3", "ANT"}], "R4": [{"CA3"}] * 4,
    "R5": [{"CA3", "ANT"}, {"ANT"}, {"ANT"}, {"ANT"}],
    "R6": [{"ANT"}, {"CA1", "CA3"}, {"CA3"}, {"CA3"}], "R7": [{"ANT"}] * 4}
TARGET = {("R1", "DG"), ("R2", "DG"), ("R2", "CA1"), ("R3", "CA3"), ("R4", "CA3"), ("R5", "ANT"),
          ("R6", "CA3"), ("R7", "ANT")}  # target stimulation groups, Figure S2 / Figure 3A boxes
span = df.groupby("group").agg(rat=("rat", "first"), kind=("kind", "first"), site=("site", "first"),
                               n=("epoch", "size"), first=("date", "min"), last=("date", "max"),
                               days=("date", "nunique"))
print(span.to_string())
for rat in sorted(FIGS2_ORDER):
    ds = ad[(ad.rat == rat) & (ad.kind == "stim")].sort_values("index").site.tolist()
    print(rat, "dandiset order", ds, "| Figure S2 order", FIGS2_ORDER[rat],
          "| same" if ds == FIGS2_ORDER[rat] else "| DIFFERENT")

### Stimulation trains and the clock check

The files carry no stimulation times. Trains of 130 Hz pulses appear as a comb of aliased lines in
the 1 kHz LFP; they are detected per 0.5 s window as runs of comb score above 1 lasting at least
4 s (`yang.find_trains`). The pulse rate of each epoch is the one, on a 0.25 Hz grid from 120 to
145 Hz, that gives the most train time; 125 Hz (fs / 8) is left out of the grid, as its harmonics
fold onto only four lines that any 8-sample periodic artifact fills (see `yang.F0_GRID`). Methods C
gives 5 s trains with 5 s pauses, which the detected trains are checked against. As the LFP is
the only data stream, the clock check asks whether the trains, which the closed-loop detector
triggers at seizures, fall inside the seizures found from the LFP, against the same with the
seizure times shifted. The main seizure detector counts train windows as ictal, so that check
would be circular; it is therefore run on the seizure found from the LFP alone, without the
trains (`sz_notrains`). To see whether the detector misses trains in groups where it finds few,
the comb score and the runs at the rat's own pulse rates that are too short to count as trains
are printed per group, and example spectrograms follow. For R7's stimulation seizures without a
train, runs of at least 2 s at R7's pulse rates are listed in a further cell.

In [ ]:
st = df[df.kind == "stim"]
print("stimulation groups: epochs with at least one train, of all epochs")
print(st.groupby("group").agg(site=("site", "first"), epochs=("epoch", "size"),
                              with_trains=("ntrains", lambda x: int((x > 0).sum())),
                              ).T.to_string())
print("pulse rate (Hz) per group, median over epochs with trains:",
      st[st.ntrains > 0].groupby("group").pulse_rate.median().to_dict())
lens = np.array([e - s for tr in st.trains for s, e in tr])
print(f"\ntrain length: n = {len(lens)}, median {np.median(lens):.2f} s, 5th to 95th percentile "
      f"{np.percentile(lens, 5):.2f} to {np.percentile(lens, 95):.2f} s")
gaps = np.array([tr[i + 1][0] - tr[i][1] for tr in st.trains for i in range(len(tr) - 1)])
print(f"pause between consecutive trains: median {np.median(gaps):.2f} s, 25th to 75th percentile "
      f"{np.percentile(gaps, 25):.2f} to {np.percentile(gaps, 75):.2f} s")
bt = df[(df.kind == "baseline") & (df.ntrains > 0)]
print(f"\nbaseline epochs with detected trains: {len(bt)} of {int((df.kind == 'baseline').sum())}")
print(bt[["group", "epoch", "pulse_rate", "trains"]].to_string())


# missed trains: the longest run of comb score above threshold at the rat's own pulse rates
rates = st[st.ntrains > 0].groupby("group").pulse_rate.median()
rat_rates = {rat: sorted(set(rates[[g for g in rates.index if g.startswith(rat)]])) for rat in df.rat.unique()}


def longest_at_rat_rate(r):
    k = np.zeros(len(yang.F0_GRID), bool)
    for f in rat_rates.get(r.rat, []):
        k |= np.abs(yang.F0_GRID - f) <= 1.0
    return r.comb_longest_by_rate[k].max() if k.any() else np.nan


df["longest_rat_rate"] = df.apply(longest_at_rat_rate, axis=1)
print("\ntrain detector diagnostics per stimulation group: epochs with trains; median over epochs of the comb "
      "score's 99th percentile; epochs whose longest run above threshold at the rat's own pulse rates "
      "(within 1 Hz) is 2 to 4 s (a train the 4 s rule may miss)")
diag = df.groupby("group").agg(
    kind=("kind", "first"), site=("site", "first"), epochs=("epoch", "size"),
    with_trains=("ntrains", lambda x: int((x > 0).sum())), comb_p99=("comb_p99", "median"),
    runs_2_to_4s=("longest_rat_rate", lambda x: int(((x >= 2) & (x < 4)).sum())))
print(diag[diag.kind == "stim"].drop(columns="kind").round(2).T.to_string())
bd = diag[diag.kind == "baseline"]
print("baseline groups: comb score median", round(bd.comb_p99.median(), 2), "(range", round(bd.comb_p99.min(), 2),
      "to", round(bd.comb_p99.max(), 2), "); baseline epochs with a 2 to 4 s run at the rat's rates:",
      int(bd.runs_2_to_4s.sum()))
print("pulse rates of each rat's stimulation groups (Hz):", rat_rates)
bt_rate = bt.apply(lambda r: any(abs(r.pulse_rate - f) <= 1.0 for f in rat_rates[r.rat]), axis=1)
print("baseline epochs whose trains are within 1 Hz of a pulse rate of the rat's stimulation groups:",
      int(bt_rate.sum()), "of", len(bt), bt[bt_rate].epoch.tolist())

df["onset_lfp"] = df.sz_notrains.apply(lambda x: x[0])
df["end_lfp"] = df.sz_notrains.apply(lambda x: x[1])
st = df[df.kind == "stim"]


def frac_in(shift, on="onset", en="end"):
    k = n = 0
    for r in st[(st.ntrains > 0) & st[on].notna()].itertuples():
        for s, e in r.trains:
            n += 1
            k += (s < getattr(r, en) + shift + 1) and (e > getattr(r, on) + shift - 1)
    return k / n


print("\nfraction of trains overlapping the seizure found from the LFP alone (trains not used), by shift "
      "of the seizure times:")
print({sh: round(frac_in(sh, "onset_lfp", "end_lfp"), 3) for sh in (-120, -60, -30, 0, 30, 60, 120)})
print("the same with the main detector, which counts trains as ictal (circular, for comparison):",
      {sh: round(frac_in(sh), 3) for sh in (-60, 0, 60)})
ok = st[(st.ntrains > 0) & st.onset_lfp.notna()]
lat2 = ok.apply(lambda r: r.trains[0][0] - r.onset_lfp, axis=1)
print(f"first train minus onset found without trains: median {lat2.median():.1f} s, range {lat2.min():.1f} "
      f"to {lat2.max():.1f}; first train inside that seizure in {int(ok.apply(lambda r: r.onset_lfp - 1 < r.trains[0][0] < r.end_lfp + 1, axis=1).sum())} "
      f"of {len(ok)} stimulated seizures; onsets of the two detectors equal in {int((ok.onset_lfp == ok.onset).sum())}")
lat = st[(st.ntrains > 0) & st.onset.notna()].apply(lambda r: r.trains[0][0] - r.onset, axis=1)
print(f"first train minus seizure onset: median {lat.median():.1f} s, 25th to 75th percentile "
      f"{lat.quantile(.25):.1f} to {lat.quantile(.75):.1f} s, range {lat.min():.1f} to {lat.max():.1f}")

In [ ]:
# example epochs: spectrogram and comb score, groups with and without detected trains
EX = [(g, f"{diag.loc[g, 'with_trains']} of {diag.loc[g, 'epochs']} epochs with trains")
      for g in ("R3-05", "R3-07", "R7-05", "R7-07")]
fig, axs = plt.subplots(len(EX), 1, figsize=(11, 2.4 * len(EX)), sharex=True)
for ax, (g, lab) in zip(axs, EX):
    gg = df[df.group == g]
    r = gg[gg.ntrains == 0].iloc[0] if (gg.ntrains == 0).any() else gg.iloc[0]
    x = yang.read_epoch(ad.set_index("group").loc[g, "asset_id"], r.epoch)
    f, t, S = yang.signal.spectrogram(x[:, CH.index(r.site)], fs=yang.FS, nperseg=500, noverlap=250)
    ax.pcolormesh(t, f, 10 * np.log10(S + 1e-12), shading="auto", cmap="viridis", vmin=-40, vmax=20)
    for s, e in r.trains:
        ax.axvspan(s, e, ymin=.96, ymax=1, color="r")
    ax.axvline(r.onset, color="w", ls="--"); ax.axvline(r.end, color="w", ls=":")
    tt, sc, f0, _ = yang.stim_score(x)
    print(f"{g} {r.epoch} ({lab}): comb score 99th percentile {np.percentile(sc, 99):.2f}, max {sc.max():.2f}, "
          f"best pulse rate {f0} Hz, trains {r.trains}")
    ax.set_title(f"{g} {r.site} stimulation, {r.epoch} ({lab}); channel {r.site}", fontsize=9)
    ax.set_ylabel("Hz")
axs[-1].set_xlabel("s in the 600 s epoch (red: detected trains; dashed: onset; dotted: end)")
plt.tight_layout()
plt.show()
print("colour scale -40 to 20 dB; values outside are clipped to the ends of the scale")

In [ ]:
# R7's stimulation epochs without a detected train: runs of comb score of at least 2 s (not 4) at the
# pulse rates of R7's stimulation groups, to see whether the 4 s rule misses trains there
R7_RATES = np.array(rat_rates["R7"])
grid7 = yang.F0_GRID[np.min(np.abs(yang.F0_GRID[:, None] - R7_RATES[None]), 1) <= 1.0]
short, keep = [], {}
for g in ("R7-01", "R7-03", "R7-05", "R7-07"):
    for r in df[(df.group == g) & (df.ntrains == 0)].itertuples():
        x = yang.read_epoch(ad.set_index("group").loc[g, "asset_id"], r.epoch)
        sc7 = yang.stim_score(x, f0_grid=grid7, min_len=2.0)
        runs7, f07 = yang.find_trains(x, min_len=2.0, score=sc7)
        ins = [bool(s < r.end + 1 and e > r.onset - 1) for s, e in runs7]
        short.append(dict(group=g, epoch=r.epoch, onset=round(r.onset, 1), end=round(r.end, 1), rate=f07,
                          runs_2s=[(round(s, 2), round(e, 2)) for s, e in runs7],
                          lengths=[round(e - s, 2) for s, e in runs7], in_seizure=sum(ins)))
        best = max([e - s for s, e in runs7], default=0)
        if best > keep.get(g, (None, -1))[1]:
            keep[g] = ((r, x, sc7, runs7), best)
SH = pd.DataFrame(short)
print(SH.to_string())
print("\nepochs without a train with a run of 2 s or more at R7's pulse rates:",
      SH.groupby("group").apply(lambda x: f"{int((x.lengths.apply(len) > 0).sum())} of {len(x)}, "
                                f"{int((x.in_seizure > 0).sum())} with a run inside the seizure").to_dict())
fig, axs = plt.subplots(len(keep), 1, figsize=(11, 2.4 * len(keep)), sharex=True)
for ax, (g, ((r, x, sc7, runs7), best)) in zip(np.atleast_1d(axs), keep.items()):
    f, t, S = yang.signal.spectrogram(x[:, CH.index(r.site)], fs=yang.FS, nperseg=500, noverlap=250)
    ax.pcolormesh(t, f, 10 * np.log10(S + 1e-12), shading="auto", cmap="viridis", vmin=-40, vmax=20)
    for s, e in runs7:
        ax.axvspan(s, e, ymin=.96, ymax=1, color="orange")
    ax.axvline(r.onset, color="w", ls="--"); ax.axvline(r.end, color="w", ls=":")
    ax.set_title(f"{g} {r.site} stimulation, {r.epoch}, no 4 s train; longest run at R7's rates {best:.2f} s", fontsize=9)
    ax.set_ylabel("Hz")
np.atleast_1d(axs)[-1].set_xlabel("s in the 600 s epoch (orange: runs of 2 s or more; dashed: onset; dotted: end)")
plt.tight_layout()
plt.show()
print("colour scale -40 to 20 dB; values outside are clipped to the ends of the scale")
del keep

In [ ]:
# example traces per rat: one stimulation seizure with trains, onset, end and trains marked
fig, axs = plt.subplots(7, 1, figsize=(11, 14))
for ax, rat in zip(axs, sorted(df.rat.unique())):
    g = df[(df.rat == rat) & (df.kind == "stim") & (df.ntrains > 0) & df.onset.notna()].iloc[0]
    x = yang.read_epoch(ad.set_index("group").loc[g.group, "asset_id"], g.epoch)
    a, b = max(0, g.onset - 30), min(600, g.end + 30)
    tt = np.arange(int(a * yang.FS), int(b * yang.FS)) / yang.FS
    for c in range(4):
        y = x[int(a * yang.FS):int(b * yang.FS), c]
        ax.plot(tt, y / (6 * x[:, c].std()) - 1.5 * c, lw=.3, color="k")
    for s, e in g.trains:
        ax.axvspan(s, e, color="r", alpha=.15)
    ax.axvline(g.onset, color="b"); ax.axvline(g.end, color="b", ls="--")
    if np.isfinite(g.onset_lfp):
        ax.axvline(g.onset_lfp, color="g", ls=":"); ax.axvline(g.end_lfp, color="g", ls=":")
    ax.set_yticks([0, -1.5, -3, -4.5], CH)
    ax.set_title(f"{g.group} {g.site} stimulation, {g.epoch}: onset {g.onset:.1f} s, end {g.end:.1f} s "
                 f"(without trains {g.onset_lfp:.1f} to {g.end_lfp:.1f} s)", fontsize=9)
axs[-1].set_xlabel("s (blue: onset and end; green dotted: the same found without trains; red: trains)")
plt.tight_layout()
plt.show()
print("each channel scaled by 6 SD of its epoch; traces beyond +-0.75 of the scale overlap the next channel")

### Seizure onset and end

The dandiset gives no seizure times, and the paper gives no rule for marking them (claims file,
opening comment, reading (b)). The assumption here (`yang.detect_seizure`): the largest over
channels of the 1-45 Hz line length in 1 s windows, divided by the channel's median over the epoch,
above 2 (with stimulation trains counted as ictal, runs merged across gaps under 7 s, and
boundaries extended while it stays above 1.5). The thresholds 1.5 and 3, and ending a seizure at
the last ictal window before a final train, are shown as variants in every duration claim.

In [ ]:
for th in yang.SZ_THRESH_VARIANTS:
    c = f"dur_{th}"
    print(f"threshold {th}: seizures found in {int(df[c].notna().sum())} of {len(df)} epochs; duration "
          f"median {df[c].median():.1f} s, range {df[c].min():.1f} to {df[c].max():.1f} s")
print("epochs without a seizure at the main threshold:", df[df.dur.isna()][["group", "epoch"]].values.tolist())
print("\nmedian duration (s) by rat and kind, main threshold:")
print(df.groupby(["rat", "kind"]).dur.median().unstack().round(1))
onsets = df.onset.dropna()
print(f"\nonset within the 600 s epoch: median {onsets.median():.1f} s, range {onsets.min():.1f} to "
      f"{onsets.max():.1f} s")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))
axs[0].hist(df.dur.dropna(), bins=40)
axs[0].set_xlabel("seizure duration (s), main threshold")
axs[0].set_ylabel("epochs")
axs[1].hist(lat, bins=30)
axs[1].set_xlabel("first train minus onset (s)")
plt.tight_layout()
plt.show()
print(f"all {int(df.dur.notna().sum())} durations ({df.dur.min():.1f} to {df.dur.max():.1f} s) and "
      f"all {len(lat)} latencies ({lat.min():.1f} to {lat.max():.1f} s) are inside the axes")

## Network claims

cf_w over the first 5 s after onset, 9 windows of 1 s with 0.5 s overlap (Methods B), from
conditional GC with BIC model order at 1 kHz (main) and, as variants, after decimation to 250 Hz
and at a fixed order of 40 at 1 kHz, where the residuals are closer to white (the BIC order leaves
them non-white). A further variant leaves out the 7 baseline seizures that carry trains.
Reading (a) of the claims file's sign (out-strength minus in-strength) is `cfw`; reading (b) is its
negative. The two readings of combining windows coincide (cf_w is linear in the GC matrix).

In [ ]:
# variant: the 7 baseline seizures that carry trains left out of the baseline sets
df["cfw_nobt"] = df.apply(lambda r: r.cfw if r.ntrains == 0 else np.nan, axis=1)
base = df[(df.kind == "baseline") & df.cfw.notna()].copy()
print("baseline seizures with cf_w:", len(base), "of", int((df.kind == "baseline").sum()))
print("VAR order (BIC, 1 kHz): median", base.gc_order.median(), "range", base.gc_order.min(), "to",
      base.gc_order.max(), "| at 250 Hz: median", base.gc_order_ds4.median())
print(f"VAR checks at 1 kHz, per seizure over its 9 windows: largest companion eigenvalue modulus median "
      f"{base.var_maxmod.median():.3f}, range {base.var_maxmod.min():.3f} to {base.var_maxmod.max():.3f}; seizures with "
      f"a window not stable (modulus >= 1): {int((base.var_maxmod >= 1).sum())} of {len(base)}")
print(f"residual whiteness (Ljung-Box, 20 lags, per channel and window): share of channel-windows with p < 0.05, "
      f"median over seizures {base.var_lb_frac.median():.2f}, mean {base.var_lb_frac.mean():.2f}, "
      f"range {base.var_lb_frac.min():.2f} to {base.var_lb_frac.max():.2f} (0.05 expected for white residuals)")
for lab, sfx in (("250 Hz, BIC order", "_ds4"), (f"1 kHz, fixed order {yang.GC_ORDER_WHITE}", "_p40")):
    mm_, lb_ = base[f"var_maxmod{sfx}"], base[f"var_lb_frac{sfx}"]
    print(f"{lab}: largest modulus median {mm_.median():.3f} (range {mm_.min():.3f} to {mm_.max():.3f}), not stable in "
          f"{int((mm_ >= 1).sum())} of {len(base)}; share of channel-windows with Ljung-Box p < 0.05: median "
          f"{lb_.median():.2f}, mean {lb_.mean():.2f}, range {lb_.min():.2f} to {lb_.max():.2f}")
GC_VARIANTS = {"1 kHz": "cfw", "250 Hz": "cfw_ds4", "1 kHz, onset threshold 1.5": "cfw_1.5",
               "1 kHz, onset threshold 3": "cfw_3.0", f"1 kHz, fixed order {yang.GC_ORDER_WHITE}": "cfw_p40",
               "1 kHz, baseline seizures with trains left out": "cfw_nobt"}
print("baseline seizures with trains left out by the last variant:", int(base.cfw_nobt.isna().sum()))


def pair_sets(V, grp, twice=False, self_pairs=False):
    R = np.corrcoef(V)
    n = len(V)
    same = grp[:, None] == grp[None, :]
    if twice:
        m = ~np.eye(n, dtype=bool)
    else:
        m = np.triu(np.ones((n, n), bool), 1)
    intra = R[m & same]
    inter = R[m & ~same]
    if self_pairs:
        intra = np.concatenate([intra, np.ones(n)])
    return intra, inter

### tables1-network-stable-short-term

> In every rat the cf_w vectors of seizure onset correlate more strongly between seizures of the
> same baseline group than between seizures of different baseline groups (Pearson r, intra-group
> against inter-group, Mann-Whitney).

Criterion: reproduced when, under one reading, intra > inter with Mann-Whitney p < 0.05 in all 7
rats and at least 10 of the 14 means lie within tolerance (Table S1 SDs); partially reproduced when
direction and significance hold in at least 5 rats. Readings: pairs once (a) or twice (b); self
pairs excluded (c) or included (d); and the GC variants.

In [ ]:
TABLE_S1 = {"R1": (.72, .30, -.11, .55), "R2": (.47, .57, .09, .56), "R3": (.77, .19, .59, .34),
            "R4": (.81, .46, .79, .46), "R5": (.74, .34, .71, .28), "R6": (.61, .34, .36, .47),
            "R7": (.81, .36, .71, .41)}
res = []
for gname, col in GC_VARIANTS.items():
    for twice, selfp in itertools.product((False, True), (False, True)):
        for rat, g in base[base[col].notna()].groupby("rat"):
            V = np.vstack(g[col].values)
            grp = np.array(g.group.tolist(), dtype=object)
            intra, inter = pair_sets(V, grp, twice, selfp)
            p = stats.mannwhitneyu(intra, inter, alternative="two-sided").pvalue
            e = TABLE_S1[rat]
            res.append(dict(gc=gname, pairs="twice" if twice else "once", self="incl" if selfp else "excl",
                            rat=rat, n=len(V), intra=intra.mean(), intra_sd=intra.std(),
                            inter=inter.mean(), inter_sd=inter.std(), p=p,
                            ok=(intra.mean() > inter.mean()) and p < 0.05,
                            in_tol=int(abs(intra.mean() - e[0]) <= e[1]) + int(abs(inter.mean() - e[2]) <= e[3])))
t1 = pd.DataFrame(res)
for selfp in ("excl", "incl"):
    print(f"1 kHz, pairs once, self pairs {selfp}:")
    print(t1[(t1.gc == "1 kHz") & (t1.pairs == "once") & (t1.self == selfp)].round(3).to_string())
summ = t1.groupby(["gc", "pairs", "self"]).agg(rats_significant=("ok", "sum"), means_in_tol=("in_tol", "sum"))
print("\nby reading (7 rats, 14 means):")
print(summ.to_string())
print("\nrats significant under each reading, listed:")
print(t1[t1.ok].groupby(["gc", "pairs", "self"]).rat.apply(lambda x: ",".join(x)).to_string())

**Result: reproduced, under reading (d) of the criterion (self pairs included).** With 1 kHz GC
and each pair once, intra-group r exceeds inter-group r with Mann-Whitney p < 0.05 in all 7 rats
and 13 of 14 means are within the Table S1 SDs (R5 inter-group 0.32 against 0.71 +- 0.28 is not).
The outcome rests on reading (d), which adds an r of 1 per seizure to the intra-group set and so
favours intra over inter: with self pairs excluded (reading (c)) the test holds in 5 rats at the
main onset (R1, R2, R3, R5, R6; R4 p = 0.88 and R7 p = 0.66, 13 of 14 means within tolerance),
which would be partially reproduced, and in 6 rats with the onset thresholds 1.5 or 3; at 250 Hz
in 2 or 3. The outcome also rests on the seizure onsets found from the LFP (an assumption). The
VAR fits are stable in all but 2 of 266 seizures, but their residuals are not white (Ljung-Box
p < 0.05 in 83% of channel-windows at the median seizure), so the BIC order leaves structure the
GC estimates do not capture. At a fixed order of 40 the residuals are close to white (median share
0.00, at most 0.33), but the largest modulus is 1.000 at the median seizure and 54 of 266
seizures have a window that is not stable; there reading (d) again holds in 7 rats, with 12 of 14
means within tolerance, and reading (c) in 4 (R1, R2, R3, R6). At 250 Hz the residuals are not
white either (median share 1.00). Leaving out the 7 baseline seizures that carry trains gives the
same counts as the main reading (7 rats under (d), 5 under (c), 13 of 14 means).

### fig2-r1-target-shifts

> In rat R1 the node with the highest cf_w at seizure onset changes across the four baseline groups
> (ANT, ANT, CA3, DG), while the cf_w pattern is consistent within each group.

Criterion: reproduced when, under one reading, the node with the highest mean cf_w in each of R1's
four baseline groups is the one named; partially when three of the four match.

In [ ]:
r1 = base[base.rat == "R1"]
named = ["ANT", "ANT", "CA3", "DG"]
for gname, col in GC_VARIANTS.items():
    for sign, lab in ((1, "out - in"), (-1, "in - out")):
        tops, means = [], []
        for gg, g in r1[r1[col].notna()].groupby("group"):
            m = sign * np.vstack(g[col].values).mean(0)
            tops.append(CH[int(np.argmax(m))])
            means.append(np.round(m, 3))
        print(f"{gname:28s} {lab}: top nodes {tops}, {sum(a == b for a, b in zip(tops, named))} of 4 match")
print("\nmean cf_w (out - in, 1 kHz) per R1 baseline group, nodes", CH)
for gg, g in r1.groupby("group"):
    V = np.vstack(g.cfw.values)
    print(gg, "n =", len(V), np.round(V.mean(0), 4), "| seizures whose top node is the group's top:",
          int((np.argmax(V, 1) == np.argmax(V.mean(0))).sum()))
fig, ax = plt.subplots(figsize=(10, 2.5))
V = np.vstack(r1.cfw.values).T
im = ax.imshow(V, aspect="auto", cmap="RdBu_r", vmin=-np.abs(V).max(), vmax=np.abs(V).max())
ax.set_yticks(range(4), CH)
b = np.cumsum(r1.groupby("group").size().values)[:-1]
for x in b:
    ax.axvline(x - .5, color="k")
plt.colorbar(im, label="cf_w (out - in)")
ax.set_xlabel("R1 baseline seizures, groups 0 to 3")
plt.show()
print(f"cf_w range shown: {V.min():.3f} to {V.max():.3f}")

**Result: reproduced.** With cf_w as out-strength minus in-strength (1 kHz GC), the top nodes of
R1's four baseline groups are ANT, ANT, CA3, DG, as named. The top node of single seizures agrees
with its group's top in 9 of 9, 8 of 10, 4 of 5 and 6 of 11 seizures. At 250 Hz three of four
match, with onset threshold 3 four, with threshold 1.5 two, at the fixed order 40 four, and with
the baseline seizures that carry trains left out four; with the sign as worded (in minus out)
none. The outcome rests on the onset assumption.

### figs2-target-assignment

> Across the 7 rats, the targets computed from each baseline group's seizures, compared with the
> site stimulated in the next group, give 8 target and 20 non-target stimulation groups as Figure
> S2 shows.

Criterion: reproduced when, under one reading, at least 24 of the 28 baseline groups give the
Figure S2 target set and the labels of all 28 stimulation groups agree with Figure S2; partially
when at least 18 baseline groups match. Readings of "significantly high": (a) the node with the
largest mean cf_w; (b) nodes whose cf_w is significantly above that of each other node
(Kruskal-Wallis, then pairwise Mann-Whitney, p < 0.05). The next group is the next file of the
dandiset.

In [ ]:
def targets_b(V, alpha=0.05):
    if stats.kruskal(*V.T).pvalue >= alpha:
        return set()
    out = set()
    for i in range(4):
        if all(V[:, i].mean() > V[:, j].mean() and
               stats.mannwhitneyu(V[:, i], V[:, j], alternative="greater").pvalue < alpha
               for j in range(4) if j != i):
            out.add(CH[i])
    return out


rows_s2 = []
for gname, col in GC_VARIANTS.items():
    for sign, lab in ((1, "out-in"), (-1, "in-out")):
        for rat in sorted(FIGS2_TARGETS):
            for bi in range(4):
                g = base[(base.rat == rat) & (base["index"] == 2 * bi) & base[col].notna()]
                V = sign * np.vstack(g[col].values)
                ta = {CH[int(np.argmax(V.mean(0)))]}
                tb = targets_b(V)
                fs2 = FIGS2_TARGETS[rat][bi]
                nxt = ad[(ad.rat == rat) & (ad["index"] == 2 * bi + 1)].site.iloc[0]
                rows_s2.append(dict(gc=gname, sign=lab, rat=rat, baseline=bi, figS2=",".join(sorted(fs2)),
                                    top=",".join(ta), sig=",".join(sorted(tb)),
                                    match_a=bool(ta <= fs2), match_b=(tb == fs2), next_site=nxt,
                                    label_a=nxt in ta, label_b=nxt in tb,
                                    label_figS2=(rat, nxt) in TARGET))
s2 = pd.DataFrame(rows_s2)
main = s2[(s2.gc == "1 kHz") & (s2.sign == "out-in")]
print(main.drop(columns=["gc", "sign"]).to_string())
print("\nby reading: baseline groups matching Figure S2 (of 28), stimulation groups labelled target "
      "(Figure S2 gives 8), labels agreeing with Figure S2 (of 28)")
print(s2.groupby(["gc", "sign"]).apply(lambda x: pd.Series(dict(
    match_a=int(x.match_a.sum()), match_b=int(x.match_b.sum()),
    targets_a=int(x.label_a.sum()), targets_b=int(x.label_b.sum()),
    labels_agree_a=int((x.label_a == x.label_figS2).sum()),
    labels_agree_b=int((x.label_b == x.label_figS2).sum())))).to_string())
print("\nFigure S2 labels applied to the dandiset's group order give", sum((r, s) in TARGET for r, s in
      ad[ad.kind == "stim"][["rat", "site"]].values), "target groups by site; Figure S2's targets applied "
      "to the dandiset's order (target when the next file's site is in the preceding baseline's "
      "Figure S2 set):", int(main.apply(lambda r: r.next_site in set(r.figS2.split(",")), axis=1).sum()))

**Result: partially reproduced.** Under reading (a), the top node, with 1 kHz GC and cf_w as
out minus in, 18 of 28 baseline groups give the Figure S2 target set (R1, R3, R4 and R7 all four;
R2 one; R5 none; R6 one), and the labels of 20 of 28 stimulation groups agree with Figure S2 (8
groups are labelled target, as in the paper, but not the same 8). At 250 Hz 18 groups match and
24 labels agree. At the fixed order 40, where the residuals are close to white, 19 groups match
and 21 labels agree; with the baseline seizures that carry trains left out, 18 and 20, as in the
main reading. Reading (b), significance, matches 15 groups at most (order 40; 12 at the BIC
order). The labels of all 28 groups
cannot agree under any reading: the dandiset's order of stimulated sites differs from Figure S2's
for R5 and R6, and Figure S2's own target sets applied to the dandiset's order label 9 groups
target, not 8.

## Seizure duration claims

Normalized duration of each seizure of a stimulation group against the preceding baseline group
(the previous file of the dandiset): (a) relative change, duration / baseline mean - 1 (main);
(b) difference from the baseline mean in s; (c) z-score against the baseline seizures. Target and
non-target labels are those of Figure S2 by rat and site, as the claims file says. Seizure sets: (a)
all seizures of the group (main); (b) only those with a detected stimulation train; and, as a
variant, all seizures of the groups with a train in at least half their seizures (R7-03, R7-05 and
R7-07 are left out). Detector variants: thresholds 1.5 and 3, and ending at the last ictal window
before a final train. Group order variants: the preceding baseline taken by the epoch dates (the
baseline group that began last on or before the stimulation group's first date; this changes only
R3-07, whose epochs are dated before R3-06's), and from Figure S2's order (which contradicts the
epoch dates for R5 and R6, shown for comparison only). Baseline variant: the 7 baseline seizures
that carry trains are left out of the baseline they normalize against.


In [ ]:
def chain_end(r, max_pause=12.0):
    """End of the seizure extended to the end of the last train in the chain that starts at the
    first train inside the seizure (consecutive trains at most `max_pause` s apart). Methods C:
    trains are "repeated while seizure activity is detected", so the device still detected the
    seizure until the last train of the chain."""
    if not np.isfinite(r.end):
        return np.nan
    tr = [t for t in r.trains if t[1] > r.onset - 1 and t[0] < r.end + 1]
    if not tr:
        return r.end
    last = tr[0]
    for t in r.trains:
        if t[0] > last[1] and t[0] - last[1] <= max_pause:
            last = t
    return max(r.end, last[1])


df["dur_trainchain"] = df.apply(lambda r: chain_end(r) - r.onset if r.kind == "stim" else r.dur, axis=1)
ext = df[(df.kind == "stim") & df.dur.notna()]
print(f"stimulation seizures whose end would move to a later train of the chain (trains at most 12 s apart): "
      f"{int((ext.dur_trainchain > ext.dur + 0.01).sum())} of {len(ext)}")

FIRST_DATE = df.groupby("group").date.min()
STIM_SHARE = df[df.kind == "stim"].groupby("group").ntrains.apply(lambda x: (x > 0).mean())


def base_by_date(rat, group):
    """Index of the baseline group of `rat` that began last on or before the stimulation group's
    first epoch date."""
    b = ad[(ad.rat == rat) & (ad.kind == "baseline")]
    b = b[b.group.map(FIRST_DATE) <= FIRST_DATE[group]]
    return int(b.sort_values("index").iloc[-1]["index"]) if len(b) else None


def norm_table(dcol="dur", seizures="all", base_order="dandiset", base_trains=True):
    out = []
    for r in ad[ad.kind == "stim"].itertuples():
        g = df[(df.group == r.group)]
        if seizures == "stimulated":
            g = g[g.ntrains > 0]
        if seizures == "groups_mostly_stimulated" and STIM_SHARE[r.group] < 0.5:
            g = g.iloc[:0]
        d = g[dcol].dropna().values
        bi = r.index - 1
        if base_order == "figS2":
            bi = 2 * FIGS2_ORDER[r.rat].index(r.site)
        elif base_order == "dates":
            bi = base_by_date(r.rat, r.group)
        bg = df[(df.rat == r.rat) & (df["index"] == bi)]
        if not base_trains:
            bg = bg[bg.ntrains == 0]
        b = bg[dcol].dropna().values
        out.append(dict(rat=r.rat, site=r.site, group=r.group, target=(r.rat, r.site) in TARGET,
                        n=len(d), base_n=len(b), base_group=f"{r.rat}-{bi:02d}", base_mean=b.mean(),
                        rel=d / b.mean() - 1, diff=d - b.mean(), z=(d - b.mean()) / b.std(ddof=1),
                        d=d, b=b))
    t = pd.DataFrame(out)
    for k in ("rel", "diff", "z"):
        t[k + "_mean"] = t[k].apply(lambda x: np.mean(x) if len(x) else np.nan)
    return t


VARIANTS = {"main": dict(), "threshold 1.5": dict(dcol="dur_1.5"), "threshold 3": dict(dcol="dur_3.0"),
            "end before final train": dict(dcol="dur_notrainend"),
            "stimulated seizures only": dict(seizures="stimulated"),
            "groups with trains in most seizures": dict(seizures="groups_mostly_stimulated"),
            "baseline by epoch dates": dict(base_order="dates"),
            "baseline in Figure S2 order": dict(base_order="figS2"),
            "baseline seizures with trains left out": dict(base_trains=False)}
NT = {k: norm_table(**v) for k, v in VARIANTS.items()}
print(NT["main"][["group", "site", "target", "n", "base_group", "base_n", "base_mean", "rel_mean", "diff_mean",
                  "z_mean"]].round(3).to_string())
m, dt = NT["main"], NT["baseline by epoch dates"]
print("\npreceding baseline by epoch dates differs from the dandiset order for:",
      [f"{a} ({b} in place of {c})" for a, b, c in zip(m.group, dt.base_group, m.base_group) if b != c])
print("R3-07 (CA1) normalized to R3-04 by dates:", dt.set_index("group").loc["R3-07", ["base_mean", "rel_mean"]].round(3).to_dict(),
      "| to R3-06 by the dandiset order:", m.set_index("group").loc["R3-07", ["base_mean", "rel_mean"]].round(3).to_dict())
print("groups left out by 'groups with trains in most seizures' (share of seizures with a train under 0.5):",
      {g: round(v, 2) for g, v in STIM_SHARE.items() if v < 0.5})
print("The Figure S2 order variant contradicts the epoch dates for R5 and R6 (see the group order check); it is "
      "shown for comparison only.")


### fig3a-target-reduces-each-rat

> In each of the 7 rats, stimulation of the target site reduced normalized seizure duration below
> the preceding baseline.

Criterion: reproduced when, under one reading, the mean normalized duration is below zero in all 8
target groups; partially in 6 or 7. The three normalizations have the same sign by construction,
so the readings that can differ are the seizure set and the detector.

In [ ]:
FIG3A = {("R1", "DG"): -.10, ("R2", "DG"): -.37, ("R2", "CA1"): -.09, ("R3", "CA3"): -.23,
         ("R4", "CA3"): -.14, ("R5", "ANT"): -.33, ("R6", "CA3"): -.21, ("R7", "ANT"): -.13}
tab = []
for k, t in NT.items():
    tt = t[t.target].set_index(["rat", "site"])
    row = {f"{a} {b}": round(tt.loc[(a, b), "rel_mean"], 3) for a, b in FIG3A}
    row["below zero (of 8)"] = int((tt.rel_mean < 0).sum())
    row["groups with a value"] = int(tt.rel_mean.notna().sum())
    tab.append(pd.Series(row, name=k))
print(pd.DataFrame(tab).T.to_string())
print("\nFigure 3A, read from the bars (not judged):", {f"{a} {b}": v for (a, b), v in FIG3A.items()})

**Result: not reproduced.** The mean normalized duration is below zero in 3 of the 8 target
groups (R3 CA3 -0.03, R6 CA3 -0.17, R7 ANT -0.20) with the main detector, and in 2 to 4 under the
variants (thresholds, end before a final train, stimulated seizures only, groups with trains in
most seizures, baseline by dates or in Figure S2 order, baseline seizures with trains left out). The durations rest on the onset and end
assumption.

### fig3a-nontarget-can-lengthen

> Stimulation of some non-target sites increased seizure duration above the preceding baseline.

Criterion: reproduced when, under one reading, at least 5 of the 20 non-target groups have a positive
mean normalized duration; not reproduced when fewer than 2 do.

In [ ]:
for k, t in NT.items():
    nt = t[~t.target]
    pos = nt[nt.rel_mean > 0]
    print(f"{k:28s}: {len(pos)} of {nt.rel_mean.notna().sum()} non-target groups positive: "
          + ", ".join(f"{r.rat} {r.site} ({r.rel_mean:+.2f})" for r in pos.itertuples()))
print("Figure 3A shows positive bars for R1 CA3 and ANT, R2 CA3, R3 DG and ANT, R4 DG and ANT, R5 CA1, "
      "R6 DG, R7 DG")

**Result: reproduced.** 13 of the 20 non-target groups have a positive mean normalized
duration with the main detector, and 10 to 13 under the variants (12 with the baseline seizures
that carry trains left out); the criterion asks for 5. Of
the 10 groups positive in Figure 3A, 8 are positive here (R1 CA3 -0.06 and R6 DG -0.28 are not).

### fig3a-per-rat-site-differences

> Within rats R2, R3, R4, R5 and R7 the four stimulation sites differ in normalized seizure duration
> (Kruskal-Wallis), while in R1 and R6 no pair of sites differs.

Criterion: per rat, Kruskal-Wallis across the four stimulation groups, then Dunn's test, (a)
Bonferroni-adjusted or (b) unadjusted. Reproduced when at least 8 of the 11 listed pairs are
significant and R1 and R6 have none; partially when at least 5 are.

In [ ]:
def dunn(samples):
    """Dunn's test after Kruskal-Wallis, with the tie correction; two-sided p per pair."""
    allv = np.concatenate(samples)
    ranks = stats.rankdata(allv)
    N = len(allv)
    _, cnt = np.unique(allv, return_counts=True)
    tie = np.sum(cnt ** 3 - cnt) / (12 * (N - 1))
    idx = np.cumsum([0] + [len(s) for s in samples])
    mr = [ranks[idx[i]:idx[i + 1]].mean() for i in range(len(samples))]
    out = {}
    for i, j in itertools.combinations(range(len(samples)), 2):
        se = np.sqrt((N * (N + 1) / 12 - tie) * (1 / len(samples[i]) + 1 / len(samples[j])))
        out[(i, j)] = 2 * stats.norm.sf(abs(mr[i] - mr[j]) / se)
    return out


LISTED = {("R2", "DG", "CA3"), ("R2", "CA3", "ANT"), ("R3", "DG", "CA1"), ("R3", "DG", "CA3"),
          ("R3", "CA3", "ANT"), ("R4", "DG", "CA1"), ("R4", "DG", "CA3"), ("R5", "CA1", "CA3"),
          ("R5", "CA1", "ANT"), ("R7", "DG", "CA3"), ("R7", "DG", "ANT")}
summary = []
for k, t in NT.items():
    lines = []
    ADJ = ("bonferroni", "unadjusted", "unadjusted, after KW p < 0.05")
    sig_listed = {a: 0 for a in ADJ}
    other = {a: [] for a in ADJ}
    for rat, g in t.groupby("rat"):
        g = g[g.n > 0]
        if len(g) < 2:
            continue
        samples = [x for x in g.rel]
        sites = g.site.tolist()
        kw = stats.kruskal(*samples).pvalue
        pd_ = dunn(samples)
        for (i, j), p in pd_.items():
            pair = (rat,) + tuple(sorted((sites[i], sites[j]), key=["DG", "CA1", "CA3", "ANT"].index))
            for adj, pp in (("bonferroni", min(1, p * len(pd_))), ("unadjusted", p),
                            ("unadjusted, after KW p < 0.05", p if kw < 0.05 else 1.0)):
                if pp < 0.05:
                    if pair in LISTED:
                        sig_listed[adj] += 1
                    else:
                        other[adj].append(f"{pair[0]} {pair[1]}-{pair[2]}")
            if k == "main":
                lines.append(f"{rat} {sites[i]:>3s}-{sites[j]:<3s} Dunn p = {p:.3g} (Bonferroni {min(1, p * 6):.3g})"
                             f"{'  listed' if pair in LISTED else ''}")
        if k == "main":
            lines.append(f"{rat} Kruskal-Wallis p = {kw:.3g}, n = {[len(s) for s in samples]}")
    if k == "main":
        print("\n".join(lines))
        main_lines = lines
    for adj in ADJ:
        summary.append(dict(variant=k, posthoc=adj, listed_significant=sig_listed[adj],
                            other_significant=", ".join(other[adj])))
print()
print(pd.DataFrame(summary).to_string())
print("\nR6, main: pairs with unadjusted Dunn p < 0.05:", [l for l in main_lines if l.startswith("R6") and "Dunn" in l
      and float(l.split("Dunn p = ")[1].split(" ")[0]) < 0.05])

**Result: partially reproduced.** With the main detector, 5 of the 11 listed pairs are
significant by unadjusted Dunn's test (reading (b): R2 ANT-CA3, R3 DG-CA1, R3 DG-CA3, R4 DG-CA1,
R4 DG-CA3) and 4 after Bonferroni adjustment (R3 DG-CA3, Bonferroni p = 0.078, drops out). None is
significant in R5 or R7 (Kruskal-Wallis p 0.94 and 0.31). R1 has no significant pair (Kruskal-Wallis
p 0.42), but R6, which the claim says has none, has two under the unadjusted reading the outcome
rests on (DG-ANT p = 0.006 and DG-CA1 p = 0.020) and one after Bonferroni adjustment (DG-ANT,
p = 0.037; Kruskal-Wallis p 0.022). The outcome sits at the threshold of 5 and rests on the
unadjusted reading. Running Dunn's test only where the Kruskal-Wallis test is significant gives
the same 5 (no listed pair is significant in R5 or R7). With the preceding baseline taken by the
epoch dates it is again 5 unadjusted and 4 adjusted, with the baseline seizures that carry trains
left out also 5 and 4, and the variants give 2 to 6 listed pairs.

### fig3a-target-vs-nontarget-total

> Pooled over rats, target-site stimulation shortened normalized seizure duration (-0.26 +- 0.03)
> significantly more than non-target stimulation (0.038 +- 0.03).

Criterion: reproduced when, under one reading, both means are within tolerance (0.03) and
Mann-Whitney p < 0.001 with target lower; partially when target is lower with p < 0.05 but a mean is
outside tolerance or p is between 0.001 and 0.05. Units: (a) seizures pooled; (b) stimulation groups.

In [ ]:
out = []
for k, t in NT.items():
    for norm in ("rel", "diff", "z"):
        T = np.concatenate(t[t.target][norm].values)
        N = np.concatenate(t[~t.target][norm].values)
        p = stats.mannwhitneyu(T, N, alternative="two-sided").pvalue
        Tg, Ng = t[t.target][norm + "_mean"].dropna(), t[~t.target][norm + "_mean"].dropna()
        pg = stats.mannwhitneyu(Tg, Ng, alternative="two-sided").pvalue
        out.append(dict(variant=k, norm=norm, target=T.mean(), target_sem=stats.sem(T), nontarget=N.mean(),
                        nontarget_sem=stats.sem(N), n_t=len(T), n_n=len(N), p_seizures=p,
                        groups_target=Tg.mean(), groups_nontarget=Ng.mean(), p_groups=pg))
tot = pd.DataFrame(out)
tot["both_in_tol"] = (abs(tot.target + .26) <= .03) & (abs(tot.nontarget - .038) <= .03)
print(tot.round(4).to_string())

**Result: not reproduced.** Pooled over seizures, target -0.026 +- 0.026 and non-target
0.066 +- 0.035 (relative change), Mann-Whitney p = 0.16: target is lower but not significantly,
and the target mean is far outside -0.26 +- 0.03. No variant gives p < 0.05 (smallest 0.10, end
before a final train; 0.18 with the baseline seizures that carry trains left out), and over groups
p is above 0.29.

### fig3a-some-nontarget-shorten

> Five non-target stimulations (R2 ANT, R3 CA1, R4 CA1, R5 CA3, R7 CA3) also significantly
> shortened seizure duration, but less than the target site of the same rat.

Criterion: readings (a) Mann-Whitney of the group's durations against the preceding baseline's;
(b) Wilcoxon signed-rank of normalized durations against zero; (c) one-sample t-test against zero.
Reproduced when, under one reading, at least 4 of the 5 named groups are significantly shorter and
in at least 4 of the 5 rats the named group's mean is above the target group's (R2: DG target);
partially when 3 of each hold.

In [ ]:
NAMED = [("R2", "ANT"), ("R3", "CA1"), ("R4", "CA1"), ("R5", "CA3"), ("R7", "CA3")]
RAT_TARGET = {"R2": "DG", "R3": "CA3", "R4": "CA3", "R5": "ANT", "R7": "ANT"}
summ = []
for k, t in NT.items():
    ti = t.set_index(["rat", "site"])
    rows_ = []
    for rat, site in NAMED:
        r = ti.loc[(rat, site)]
        if r.n < 2:
            rows_.append(dict(group=f"{rat} {site}", n=r.n))
            continue
        pa = stats.mannwhitneyu(r.d, r.b, alternative="less").pvalue
        pb = stats.wilcoxon(r.rel, alternative="less").pvalue
        pc = stats.ttest_1samp(r.rel, 0, alternative="less").pvalue
        tm = ti.loc[(rat, RAT_TARGET[rat]), "rel_mean"]
        rows_.append(dict(group=f"{rat} {site}", n=r.n, mean=r.rel_mean, p_a=pa, p_b=pb, p_c=pc,
                          target_mean=tm, less_effective=r.rel_mean > tm))
    x = pd.DataFrame(rows_)
    if k == "main":
        print(x.round(4).to_string())
    summ.append(dict(variant=k, sig_a=int((x.p_a < .05).sum()), sig_b=int((x.p_b < .05).sum()),
                     sig_c=int((x.p_c < .05).sum()), less_effective_than_target=int(x.less_effective.sum())))
print()
print(pd.DataFrame(summ).to_string())
others = []
t = NT["main"]
for r in t[~t.target].itertuples():
    if (r.rat, r.site) not in NAMED and r.n >= 2:
        if stats.mannwhitneyu(r.d, r.b, alternative="less").pvalue < .05:
            others.append(f"{r.rat} {r.site}")
print("other non-target groups significantly shorter than baseline by reading (a), main:", others)

**Result: not reproduced.** With the main detector, 3 of the 5 named groups are significantly
shorter than baseline by readings (a) and (c) (R2 ANT, R3 CA1, R7 CA3) and 2 by (b) (R2 ANT, R3
CA1), but the named group's mean is above the target group's in only 2 of 5 rats (R4, R5); in
R2, R3 and R7 the named group is shorter than the target group. No variant reaches 3 and 3. R6 DG
is the one other non-target group significantly shorter than baseline by reading (a).

### fig3b-same-site-target-vs-nontarget

> For the same site, stimulation as a target shortened normalized seizure duration more than
> stimulation of that site as a non-target, significantly for DG, CA3 and ANT but not CA1.

Criterion: per site, per-seizure normalized durations of target groups against non-target groups
pooled over rats, Mann-Whitney (unpaired t-test as a variant). Reproduced when, under one
normalization reading, DG, CA3 and ANT each show target lower with p < 0.05; partially for two.

In [ ]:
out = []
for k, t in NT.items():
    for norm in ("rel", "diff", "z"):
        for site in ("DG", "CA1", "CA3", "ANT"):
            g = t[t.site == site]
            T = np.concatenate(g[g.target][norm].values)
            N = np.concatenate(g[~g.target][norm].values)
            ok_ = len(T) > 1 and len(N) > 1
            out.append(dict(variant=k, norm=norm, site=site, target=T.mean() if len(T) else np.nan,
                            nontarget=N.mean(), n_t=len(T), n_n=len(N),
                            p_mw=stats.mannwhitneyu(T, N, alternative="two-sided").pvalue if ok_ else np.nan,
                            p_t=stats.ttest_ind(T, N).pvalue if ok_ else np.nan))
f3b = pd.DataFrame(out)
f3b["target_lower_sig"] = (f3b.target < f3b.nontarget) & (f3b.p_mw < 0.05)
print(f3b[f3b.variant == "main"].round(4).to_string())
print()
print(f3b.groupby(["variant", "norm"]).apply(
    lambda x: ",".join(x[x.target_lower_sig & x.site.isin(["DG", "CA3", "ANT"])].site)).to_string())

**Result: not reproduced.** Only ANT shows target lower with p < 0.05 (relative change, main:
-0.11 against 0.11, p = 0.035); DG (p = 0.51) and CA3 (p = 0.13; t-test p = 0.050) do not, and in
DG the target mean is lower without significance (0.09 against 0.19). CA1 does not differ either
(p = 0.20). Other normalizations and variants give at most one of the three sites (ANT or CA3).

## PSD claims

PSD ratio per band and channel in the pre and post windows of Methods D (stimulation groups: onset
to the first train, then the 5 s after the first train; baseline: 0-5 s and 10-15 s after onset).
In the main window the post window ends where the next train starts, when that is within the 5 s
(an assumption: otherwise the next train's artifact, whose aliases fall in gamma, high gamma and
the HFO bands, enters the window). Window variants: (w1) the post window as worded, 5 s whatever
follows; (w2) as worded, leaving out the seizures whose next train starts within it; (w3) the main
window with the pre window cut to the last 5 s before the first train. Stimulation seizures without
a train after onset have no window and are left out. Conditions: sham (baseline seizures), target
and non-target. Readings: change measure (a) (post - pre) / pre, (b) log2(post / pre), (c) post -
pre; Welch segments 1 s or 0.5 s; total range 0.5-120 Hz or 0-500 Hz; channels all four (a), or
the stimulated channel only for stimulation groups (b); post hoc Dunn Bonferroni or unadjusted.
The window variants are run with 1 s segments and 0.5-120 Hz. Unit: seizure-channel pairs.

In [ ]:
BANDS = list(yang.BANDS)
psd = df[df.psd_pre_1000.notna()].copy()
psd["cond"] = psd.apply(lambda r: "sham" if r.kind == "baseline" else
                        ("target" if (r.rat, r.site) in TARGET else "nontarget"), axis=1)
print("seizures with PSD windows:", psd.groupby("cond").size().to_dict())
print("by rat:", psd.groupby(["rat", "cond"]).size().unstack().fillna(0).astype(int).to_dict())
sp = psd[psd.kind == "stim"]
prel = sp.psd_windows.apply(lambda w: w[0][1] - w[0][0])
postl = sp.psd_windows.apply(lambda w: w[1][1] - w[1][0])
print(f"\nstimulation seizures: pre window (onset to first train) median {prel.median():.1f} s, 5th to 95th "
      f"percentile {prel.quantile(.05):.1f} to {prel.quantile(.95):.1f} s, range {prel.min():.1f} to {prel.max():.1f}; "
      f"under 1 s: {int((prel < 1).sum())}, 1 to 2 s: {int(((prel >= 1) & (prel < 2)).sum())}, over 5 s: {int((prel > 5).sum())} "
      f"of {len(prel)} (pre windows under 1 s use a Welch segment as long as the window)")
print(f"next train starting within the 5 s after the first: {int(sp.next_train_in_post.sum())} of {len(sp)} stimulation "
      f"seizures ({sp.next_train_in_post.mean():.0%}); main post window length median {postl.median():.2f} s, "
      f"range {postl.min():.2f} to {postl.max():.2f}, under 5 s in {int((postl < 4.99).sum())}")
print("by condition:", sp.groupby("cond").next_train_in_post.agg(["sum", "size"]).to_dict("index"))

WKEY = {"main": "", "w1 paper": "_paper", "w2 paper, no overlap": "_paper", "w3 pre 5 s": "_pre5"}


def psd_change(band, measure, nps, total, channels, window="main"):
    sfx = WKEY[window]
    key_pre = f"psd_pre_{nps}" + ("_total500" if total == "0-500" else "") + sfx
    key_post = f"psd_post_{nps}" + ("_total500" if total == "0-500" else "") + sfx
    out = {}
    for c, g in psd.groupby("cond"):
        if window == "w2 paper, no overlap":
            g = g[~g.next_train_in_post.astype(bool)]
        vals = []
        for r in g.itertuples():
            pre, post = getattr(r, key_pre), getattr(r, key_post)
            if not isinstance(pre, dict):
                continue
            pre, post = pre[band], post[band]
            chs = range(4) if (channels == "all" or c == "sham") else [CH.index(r.site)]
            for ch in chs:
                a, b = pre[ch], post[ch]
                vals.append((b - a) / a if measure == "rel" else np.log2(b / a) if measure == "log2" else b - a)
        out[c] = np.array(vals)
    return out


READ = [r + ("main",) for r in itertools.product(("rel", "log2", "diff"), (1000, 500), ("0.5-120", "0-500"),
                                                 ("all", "stimulated"))]
READ = [r for r in READ if not (r[1] == 500 and r[2] == "0-500")]  # 0-500 Hz total computed with 1 s segments
READ += [r + (w,) for r in itertools.product(("rel", "log2", "diff"), (1000,), ("0.5-120",), ("all", "stimulated"))
         for w in ("w1 paper", "w2 paper, no overlap", "w3 pre 5 s")]
prow = []
for measure, nps, total, chans, window in READ:
    for band in BANDS:
        x = psd_change(band, measure, nps, total, chans, window)
        S, Tt, Nn = x["sham"], x["target"], x["nontarget"]
        kw = stats.kruskal(S, Tt, Nn).pvalue
        dn = dunn([Tt, S, Nn])
        prow.append(dict(window=window, measure=measure, nperseg=nps, total=total, channels=chans, band=band,
                         sham=np.mean(S), target=np.mean(Tt), nontarget=np.mean(Nn),
                         sham_med=np.median(S), target_med=np.median(Tt), nontarget_med=np.median(Nn),
                         kw_p=kw, p_t_vs_sham=dn[(0, 1)], p_t_vs_nt=dn[(0, 2)],
                         n=(len(S), len(Tt), len(Nn))))
P = pd.DataFrame(prow)
MAIN = (P.window == "main") & (P.measure == "rel") & (P.nperseg == 1000) & (P.total == "0.5-120") & (P.channels == "all")
print(P[MAIN].round(4).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
m = P[(P.window == "main") & (P.measure == "log2") & (P.nperseg == 1000) & (P.total == "0.5-120") & (P.channels == "all")]
for i, c in enumerate(("sham", "target", "nontarget")):
    ax.bar(np.arange(6) + i * .27, m[c], width=.27, label=c)
ax.set_xticks(np.arange(6) + .27, BANDS)
ax.set_ylabel("mean log2(post / pre) PSD ratio")
ax.legend()
plt.show()
print("plotted means range", round(m[["sham", "target", "nontarget"]].min().min(), 3), "to",
      round(m[["sham", "target", "nontarget"]].max().max(), 3))

### fig4b-psd-target-bands

> After target stimulation the change in PSD ratio was larger in low bands (alpha, and theta or
> beta) and more negative in gamma than after sham (baseline) or non-target stimulation.

Criterion: reproduced when, under one reading of the method and one of the bands ((i) alpha and
theta up, gamma down; (ii) alpha and beta up, gamma down), target differs from both sham and
non-target in the named direction with p < 0.05 (Kruskal-Wallis, then the post hoc test) in all
three bands; partially when alpha and gamma hold.

In [ ]:
def holds(r, up, adj):
    k = 3 if adj == "bonferroni" else 1
    if r.kw_p >= 0.05:
        return False
    if up:
        d = r.target > r.sham and r.target > r.nontarget
    else:
        d = r.target < r.sham and r.target < r.nontarget
    # direction judged on medians as well as means, as the test is rank based
    if up:
        d = d and r.target_med > r.sham_med and r.target_med > r.nontarget_med
    else:
        d = d and r.target_med < r.sham_med and r.target_med < r.nontarget_med
    return d and min(1, r.p_t_vs_sham * k) < 0.05 and min(1, r.p_t_vs_nt * k) < 0.05


ev = []
for (window, measure, nps, total, chans), g in P.groupby(["window", "measure", "nperseg", "total", "channels"]):
    gb = g.set_index("band")
    for adj in ("bonferroni", "unadjusted"):
        h = {b: holds(gb.loc[b], b != "gamma", adj) for b in ("theta", "alpha", "beta", "gamma")}
        ev.append(dict(window=window, measure=measure, nperseg=nps, total=total, channels=chans, posthoc=adj, **h,
                       reading_i=h["theta"] and h["alpha"] and h["gamma"],
                       reading_ii=h["beta"] and h["alpha"] and h["gamma"],
                       alpha_and_gamma=h["alpha"] and h["gamma"]))
E = pd.DataFrame(ev)
print(E.to_string())
print("\nreadings where all three bands hold:", int((E.reading_i | E.reading_ii).sum()), "of", len(E),
      "| alpha and gamma hold:", int(E.alpha_and_gamma.sum()))
print("by window:", E.groupby("window").apply(lambda x: f"{int((x.reading_i | x.reading_ii).sum())} of {len(x)} "
      f"(alpha and gamma {int(x.alpha_and_gamma.sum())}); band holding in any reading: "
      + ",".join(b for b in ("theta", "alpha", "beta", "gamma") if x[b].any())).to_dict())
print("\nalpha and gamma per reading (KW p, target vs sham p, target vs non-target p, Dunn unadjusted):")
print(P[P.band.isin(["alpha", "gamma", "theta", "beta"])][["window", "measure", "nperseg", "total", "channels", "band",
      "sham_med", "target_med", "nontarget_med", "kw_p", "p_t_vs_sham", "p_t_vs_nt"]].round(4).to_string())

# relative change and log2 ratio are monotone in post / pre, so their ranks, and so their KW and Dunn p values, agree
kk = ["window", "nperseg", "total", "channels", "band"]
a_ = P[P.measure == "rel"].set_index(kk)[["kw_p", "p_t_vs_sham", "p_t_vs_nt"]]
b_ = P[P.measure == "log2"].set_index(kk)[["kw_p", "p_t_vs_sham", "p_t_vs_nt"]].loc[a_.index]
print(f"\nrel and log2 readings with the same KW and Dunn p values: {int(np.isclose(a_, b_, rtol=1e-9).all(1).sum())} "
      f"of {len(a_)} (band x method); distinct readings of window, method and post hoc test: "
      f"{len(E[E.measure != 'log2'])} of {len(E)}")
print("readings holding a band, by band, as in the table above (unadjusted, stimulated channel, 0-500 Hz, diff):",
      E[(E.window == "main") & (E.measure == "diff") & (E.total == "0-500") & (E.channels == "stimulated")
        & (E.posthoc == "unadjusted")][["theta", "alpha", "beta", "gamma"]].to_dict("records"))
print(P[(P.window == "main") & (P.measure == "diff") & (P.total == "0-500") & (P.channels == "stimulated")
        & P.band.isin(["alpha", "gamma"])][["band", "sham_med", "target_med", "nontarget_med", "kw_p", "p_t_vs_sham",
                                            "p_t_vs_nt"]].round(4).to_string())

# variant: the 7 baseline seizures that carry trains left out of the sham set (main method reading)
psd_all = psd
psd = psd_all[~((psd_all.kind == "baseline") & (psd_all.ntrains > 0))]
print(f"\nsham set without the baseline seizures that carry trains: {int((psd.cond == 'sham').sum())} seizures")
rows_nb = []
for band in BANDS:
    x = psd_change(band, "rel", 1000, "0.5-120", "all")
    dn = dunn([x["target"], x["sham"], x["nontarget"]])
    rows_nb.append(dict(band=band, sham_med=np.median(x["sham"]), target_med=np.median(x["target"]),
                        nontarget_med=np.median(x["nontarget"]), sham=np.mean(x["sham"]), target=np.mean(x["target"]),
                        nontarget=np.mean(x["nontarget"]),
                        kw_p=stats.kruskal(x["sham"], x["target"], x["nontarget"]).pvalue,
                        p_t_vs_sham=dn[(0, 1)], p_t_vs_nt=dn[(0, 2)]))
psd = psd_all
PN = pd.DataFrame(rows_nb).set_index("band")
print(PN.round(4).to_string())
print("bands holding (unadjusted):", {b: bool(holds(PN.loc[b], b != "gamma", "unadjusted")) for b in ("theta", "alpha", "beta", "gamma")})

**Result: not reproduced.** In none of the 72 readings of window, method and post hoc test do the
three bands hold, nor alpha and gamma together. The relative change and the log2 ratio give the
same Kruskal-Wallis and Dunn p values (72 of 72 band and method readings), as both are monotone in
post / pre, so only 48 of the 72 readings are distinct tests. With the main window, relative change, 1 s
segments, 0.5-120 Hz and all channels, theta falls after target stimulation (median -0.41,
against -0.04 sham and -0.33 non-target), alpha differs between conditions (Kruskal-Wallis
p < 0.001) but the target median (0.48) lies between sham (0.62) and non-target (0.25), beta
differs (p = 0.044) with target not different from sham (Dunn p = 0.80), and gamma does not differ
(p = 0.17). With the difference measure beta rises after target stimulation against both sham and
non-target (stimulated channel, Dunn p 0.0004 and 0.0013); alpha holds in no reading. Gamma holds
in one reading, the difference measure on the stimulated channel
with the 0-500 Hz total and unadjusted Dunn tests (Kruskal-Wallis p = 0.036, target against sham
p = 0.011 and against non-target p = 0.022), where alpha does not (p = 0.53); with 0.5-120 Hz it
does not. Leaving out of the sham set the 7 baseline seizures that carry trains, no band holds in
the main reading. The post
window overlaps the next train in 48 of 228 stimulation seizures as worded; cutting it there
(main), leaving those seizures out, or using a 5 s pre window changes none of this. The windows
rest on the trains found here; R7 contributes 3 target and 9 non-target seizures.

### fig4b-psd-other-bands-unchanged

> In the other bands (delta and high gamma, and whichever of theta and beta is not named as
> changed) the change in PSD ratio did not differ between conditions.

Criterion: a claim of absence, judged on Kruskal-Wallis across sham, target and non-target at
p < 0.05. Delta and high gamma are judged in every reading; beta under band reading (i) and theta
under (ii). fig4b-psd-target-bands was decided by neither band reading, so both are shown, under the
main method reading ((a) relative change, 1 s segments, 0.5-120 Hz, all channels).

In [ ]:
mm = P[MAIN].set_index("band")
for b in ("delta", "high_gamma", "beta", "theta"):
    r = mm.loc[b]
    print(f"{b:10s} KW p = {r.kw_p:.3g}; mean change sham {r.sham:+.3f}, target {r.target:+.3f}, "
          f"non-target {r.nontarget:+.3f}; medians {r.sham_med:+.3f}, {r.target_med:+.3f}, {r.nontarget_med:+.3f}")
print("\nKW p for delta and high gamma under every method reading:")
print(P[P.band.isin(["delta", "high_gamma"])].pivot_table(index=["window", "measure", "nperseg", "total", "channels"],
      columns="band", values="kw_p").to_string())
kwp = P[P.band.isin(["delta", "high_gamma"])].pivot_table(index=["window", "measure", "nperseg", "total", "channels"],
                                                           columns="band", values="kw_p")
print(f"method readings with KW p < 0.05: delta {int((kwp.delta < .05).sum())} of {len(kwp)}, high gamma "
      f"{int((kwp.high_gamma < .05).sum())} of {len(kwp)}; with all four channels: delta "
      f"{int((kwp.xs('all', level='channels').delta < .05).sum())} of {len(kwp.xs('all', level='channels'))}")
# effect size: epsilon squared of the KW test
for b in ("delta", "high_gamma", "beta", "theta"):
    x = psd_change(b, "rel", 1000, "0.5-120", "all")
    H = stats.kruskal(*x.values()).statistic
    n = sum(len(v) for v in x.values())
    print(f"{b}: KW H = {H:.1f}, epsilon squared = {H / (n - 1):.3f}")
print("\nwithout the baseline seizures that carry trains (main method reading): KW p",
      {b: float(f"{PN.loc[b, 'kw_p']:.3g}") for b in ("delta", "high_gamma", "beta", "theta")})

**Result: not reproduced.** Delta differs between conditions under the main reading
(Kruskal-Wallis p = 1.5e-15, epsilon squared 0.035) and in 29 of the 36 method readings (all 18
with all four channels), and high gamma under the main reading (p = 0.0026, epsilon squared 0.006)
and 31 of 36. Beta, judged under band reading (i), also differs (p = 0.044); theta, judged under
(ii), does too (p = 5.9e-12). With the 7 baseline seizures that carry trains left out of the sham
set, delta (p = 2.9e-16), high gamma (p = 0.0025) and theta still differ, and beta does not
(p = 0.062).

## fig5b-iis-post-seizure

> The IIS rate 15 s to 45 s after seizures was lower after target stimulation than after
> non-target stimulation.

Criterion: IIS detected as in Methods F on all channels in the 180 s before onset and after the
end, rate per second. Baseline for the 5 SD threshold: (a) the 180 s pre-seizure stretch (the
interictal segments of reading (b) are not in the dandiset). Tests: (a) Mann-Whitney of per
seizure-channel mean rates over 15-45 s; (b) paired t-test across time bins of 15-45 s (1 s or
5 s bins), target curve against non-target curve. Reproduced when target is lower with p < 0.001;
partially with p < 0.05. Bins after the end of the 600 s epoch are missing and left out; times
inside a stimulation train are not searched.

In [ ]:
ii = df[df.iis_post.notna()].copy()
ii["cond"] = ii.apply(lambda r: "sham" if r.kind == "baseline" else
                      ("target" if (r.rat, r.site) in TARGET else "nontarget"), axis=1)
curves = {}
win_rate = {}
for c, g in ii.groupby("cond"):
    post = np.vstack(list(g.iis_post))  # seizure-channel x 180 bins of 1 s
    pre = np.vstack(list(g.iis_pre))
    curves[c] = (np.nanmean(pre, 0)[::-1], np.nanmean(post, 0))
    win_rate[c] = np.nanmean(post[:, 15:45], 1)
    print(f"{c:9s}: {len(g)} seizures, {post.shape[0]} seizure-channels; rate 15-45 s after end "
          f"{np.nanmean(win_rate[c]):.3f} /s (SEM {stats.sem(win_rate[c], nan_policy='omit'):.3f}); "
          f"180 s before onset {np.nanmean(pre):.3f} /s; missing bins in 15-45 s: {int(np.isnan(post[:, 15:45]).sum())}")
T, N = win_rate["target"], win_rate["nontarget"]
T, N = T[~np.isnan(T)], N[~np.isnan(N)]
print(f"\n(a) Mann-Whitney target vs non-target, seizure-channels: p = {stats.mannwhitneyu(T, N).pvalue:.3g} "
      f"(target {T.mean():.3f}, non-target {N.mean():.3f} /s, difference {T.mean() - N.mean():+.3f})")
for bw in (1, 5):
    ct = curves["target"][1][15:45].reshape(-1, bw).mean(1)
    cn = curves["nontarget"][1][15:45].reshape(-1, bw).mean(1)
    print(f"(b) paired t-test over {len(ct)} bins of {bw} s: p = {stats.ttest_rel(ct, cn).pvalue:.3g}, "
          f"mean difference {np.mean(ct - cn):+.3f} /s")
# per seizure (mean over channels), and per rat
segs = {c: g.apply(lambda r: np.nanmean(r.iis_post[:, 15:45]), axis=1) for c, g in ii.groupby("cond")}
print(f"per seizure (channel mean): target {segs['target'].mean():.3f}, non-target {segs['nontarget'].mean():.3f}, "
      f"Mann-Whitney p = {stats.mannwhitneyu(segs['target'].dropna(), segs['nontarget'].dropna()).pvalue:.3g}")
ii["w"] = ii.apply(lambda r: np.nanmean(r.iis_post[:, 15:45]), axis=1)
print("\nper rat, mean rate 15-45 s (/s):")
pr = ii.groupby(["rat", "cond"]).w.mean().unstack()
print(pr.round(3).to_string())
print(f"rats with target below non-target: {int((pr.target < pr.nontarget).sum())} of {len(pr)}; paired t-test over "
      f"rats (rat as the unit, not a reading of the criterion): p = {stats.ttest_rel(pr.target, pr.nontarget).pvalue:.3g}; "
      f"Wilcoxon p = {stats.wilcoxon(pr.target, pr.nontarget).pvalue:.3g}")
for bw in (1, 5):
    ct = curves["target"][1][15:45].reshape(-1, bw).mean(1)
    cn = curves["nontarget"][1][15:45].reshape(-1, bw).mean(1)
    d_ = ct - cn
    print(f"lag-1 autocorrelation of the target minus non-target difference over {len(d_)} bins of {bw} s: "
          f"{np.corrcoef(d_[:-1], d_[1:])[0, 1]:.2f}")
fig, ax = plt.subplots(figsize=(9, 3))
for c, (pre, post) in curves.items():
    ax.plot(np.arange(-180, 0) + .5, pre, lw=.8, label=c)
    ax.plot(np.arange(0, 180) + .5, post, lw=.8, color=ax.lines[-1].get_color())
ax.axvspan(15, 45, color="0.9")
ax.set_xlabel("s before onset (left) and after end (right), the seizure itself removed")
ax.set_ylabel("IIS rate (/s)")
ax.legend()
plt.show()
print("curve range:", round(min(np.nanmin(v) for p in curves.values() for v in p), 3), "to",
      round(max(np.nanmax(v) for p in curves.values() for v in p), 3))

**Result: reproduced, under reading (b) of the test.** The IIS rate 15-45 s after the seizure is
0.141 /s after target and 0.212 /s after non-target stimulation (sham 0.196). By reading (b), the
paired t-test over 30 bins of 1 s, p = 8.5e-10, and over 6 bins of 5 s p = 0.0017; by reading
(a), Mann-Whitney over seizure-channels, p = 0.0013, which alone would be partially reproduced;
per seizure p = 0.016. Reading (b), which the criterion names, treats the time bins of the two
mean curves as independent samples, although neighbouring bins are correlated (lag-1
autocorrelation of the difference 0.40 at 1 s). Target is lower than non-target in 5 of 7 rats
(not R3 and R4), and with the rat as the unit (not a reading of the criterion) p = 0.23. The
outcome rests on the seizure end found from the LFP and on the pre-seizure stretch as the
threshold baseline.

## Ictal HFO claims

Ripple (80-250 Hz) and fast ripple (250 to just below 500 Hz) detections with the HFO App's short
line length detector (Methods E, ref. 50) at the defaults of the app's detector dialog, read from
its source (github.com/zelanolab/hfoapp, commit 1ccff19; `yang.sll_energy`): FIR band-pass, 5 ms
line length, threshold the 97.5th percentile of the line length over the loaded epoch, events
longer than 12 ms. No manual review (it cannot be repeated). Windows as for the PSD claims (main:
post window cut at the next train; (w1) as worded). Threshold reference: (r1) the whole 600 s
epoch, the app's default when the file epoch is loaded (main); (r2) the epoch outside the
stimulation trains; (r3) the 180 s before onset. Normalized to pre-stimulation as relative change
of the rate. Pairing: (a) by rat, the target groups' mean against the mean of the non-target
groups (and of the baseline groups); (b) unpaired t-test over groups. Channels: (a) all four; (b)
the stimulated channel (all four for baseline). The group value is the group's mean post rate over
its mean pre rate, minus 1, so that windows with no pre event do not divide by zero; the median
over seizure-channels of the per-window relative change (where the pre rate is above zero) is
printed beside it.

In [ ]:
hf = df[df["hfo_ripple_main_epoch"].notna()].copy()
hf["cond"] = hf.apply(lambda r: "baseline" if r.kind == "baseline" else
                      ("target" if (r.rat, r.site) in TARGET else "nontarget"), axis=1)
print("seizures with HFO windows:", hf.groupby("cond").size().to_dict())
HREF = {"r1 epoch": "epoch", "r2 epoch outside trains": "epoch_notrains", "r3 pre-seizure 180 s": "preseizure"}
HWIN = {"main": "main", "w1 paper": "paper"}


def hfo_groups(nm, window, ref, chans):
    key = f"hfo_{nm}_{HWIN[window]}_{HREF[ref]}"
    out = []
    for gname, g in hf[hf[key].notna()].groupby("group"):
        r0 = g.iloc[0]
        chs = list(range(4)) if (chans == "all" or r0.kind == "baseline") else [CH.index(r0.site)]
        pre = np.concatenate([x["pre"][chs] for x in g[key]])
        post = np.concatenate([x["post"][chs] for x in g[key]])
        k = pre > 0
        out.append(dict(group=gname, rat=r0.rat, cond=r0.cond, n=len(g), pre=pre.mean(), post=post.mean(),
                        change=post.mean() / pre.mean() - 1 if pre.mean() > 0 else np.nan,
                        median_window_change=np.median(post[k] / pre[k] - 1) if k.any() else np.nan))
    return pd.DataFrame(out)


def hfo_tests(nm, window="main", ref="r1 epoch", chans="all"):
    G = hfo_groups(nm, window, ref, chans)
    per_rat = G.groupby(["rat", "cond"]).change.mean().unstack()
    res = dict(band=nm, window=window, reference=ref, channels=chans)
    for c in ("target", "nontarget", "baseline"):
        res[c] = G[G.cond == c].change.mean()
        res[c + "_median_window"] = G[G.cond == c].median_window_change.median()
    for other in ("nontarget", "baseline"):
        pr = per_rat[["target", other]].dropna()
        res[f"paired_p_vs_{other}"] = stats.ttest_rel(pr.target, pr[other]).pvalue if len(pr) > 1 else np.nan
        res[f"n_rats_vs_{other}"] = len(pr)
        a, b = G[G.cond == "target"].change.dropna(), G[G.cond == other].change.dropna()
        res[f"unpaired_p_vs_{other}"] = stats.ttest_ind(a, b).pvalue
    return res, G, per_rat


H = pd.DataFrame([hfo_tests(nm, w, ref, ch)[0] for nm in ("ripple", "fr") for w in HWIN for ref in HREF
                  for ch in ("all", "stimulated")])
print(H.round(4).to_string())
print("\nmean HFO rate (/s) in the pre and post windows, main reading, all channels:")
for nm in ("ripple", "fr"):
    G = hfo_groups(nm, "main", "r1 epoch", "all")
    print(nm, G.groupby("cond")[["pre", "post"]].mean().round(3).to_dict("index"))

### fig6e-ictal-ripples-no-difference

> During seizures, ripple rates normalized to pre-stimulation did not differ between target and
> non-target stimulation.

Criterion: a claim of absence, the paired t-test at p < 0.05, target against non-target; reproduced
when not significant under pairing (a); not reproduced when it is.

In [ ]:
res, G, per_rat = hfo_tests("ripple")
print("main reading (main window, threshold over the epoch, all channels), per rat:")
print(per_rat.round(3).to_string())
print(f"\npaired t-test by rat, target vs non-target: p = {res['paired_p_vs_nontarget']:.3g} over "
      f"{res['n_rats_vs_nontarget']} rats; target {res['target']:+.3f}, non-target {res['nontarget']:+.3f} "
      f"(difference {res['target'] - res['nontarget']:+.3f}); unpaired over groups p = {res['unpaired_p_vs_nontarget']:.3g}")
print("groups without a value (no pre event or no window):", G[G.change.isna()].group.tolist())
print("median over seizure-channels of the window change: target", round(res["target_median_window"], 3),
      "non-target", round(res["nontarget_median_window"], 3), "baseline", round(res["baseline_median_window"], 3))
rp = H[H.band == "ripple"]
print("paired p by rat, target vs non-target, over the", len(rp), "readings: range",
      round(rp.paired_p_vs_nontarget.min(), 3), "to", round(rp.paired_p_vs_nontarget.max(), 3), "; significant in",
      int((rp.paired_p_vs_nontarget < .05).sum()))
print("Figure 6E (read from the plot, not judged): target about -0.16, non-target about -0.04")

**Result: reproduced, under the all-channels reading.** With the HFO App's detector defaults, the
main window and all four channels, the paired t-test by rat does not find a difference in
normalized ictal ripple rate between target and non-target stimulation (p = 0.061, 7 rats; target
+1.46, non-target +4.22; unpaired over groups p = 0.19). The result is weak: 5 of the 12 method
readings are significant with target lower, among them the stimulated channel at the main window
and threshold (p = 0.037), and the median per-window change is -0.75 after target and +1.22 after
non-target stimulation. The group means rise from the pre to the post window in every condition,
unlike Figure 6E (about -0.16 and -0.04). No manual review was done.

### fig6f-ictal-fast-ripples-reduced

> During seizures, target stimulation reduced the fast ripple rate, normalized to pre-stimulation,
> compared with baseline and with non-target stimulation.

Criterion: reproduced when, under one pairing reading, the target mean is below both others with
p < 0.05 for each comparison; partially when one comparison holds.

In [ ]:
res, G, per_rat = hfo_tests("fr")
print("main reading (main window, threshold over the epoch, all channels), per rat:")
print(per_rat.round(3).to_string())
for other in ("baseline", "nontarget"):
    print(f"target vs {other}: target {res['target']:+.3f}, {other} {res[other]:+.3f}; paired by rat p = "
          f"{res[f'paired_p_vs_{other}']:.3g} ({res[f'n_rats_vs_{other}']} rats); unpaired p = "
          f"{res[f'unpaired_p_vs_{other}']:.3g}")
fr = H[H.band == "fr"].copy()
fr["below_both"] = (fr.target < fr.baseline) & (fr.target < fr.nontarget)
fr["holds_both"] = fr.below_both & (fr.paired_p_vs_baseline < .05) & (fr.paired_p_vs_nontarget < .05)
fr["holds_one"] = ((fr.target < fr.baseline) & (fr.paired_p_vs_baseline < .05)) | \
    ((fr.target < fr.nontarget) & (fr.paired_p_vs_nontarget < .05))
print("\nall fast ripple readings:")
print(fr.round(4).to_string())
print("readings with target below both:", int(fr.below_both.sum()), "of", len(fr), "| both comparisons significant:",
      int(fr.holds_both.sum()), "| one comparison significant with target lower:", int(fr.holds_one.sum()))
print("median over seizure-channels of the window change, main: target", round(res["target_median_window"], 3),
      "baseline", round(res["baseline_median_window"], 3), "non-target", round(res["nontarget_median_window"], 3))
print("Figure 6F (read from the plot, not judged): target about -0.25, baseline about 0.2, non-target about 0.1")
hf_all = hf
hf = hf_all[~((hf_all.kind == "baseline") & (hf_all.ntrains > 0))]
res_nb, _, _ = hfo_tests("fr")
hf = hf_all
print(f"\nbaseline seizures with trains left out (main reading): target {res_nb['target']:+.3f}, baseline "
      f"{res_nb['baseline']:+.3f}, non-target {res_nb['nontarget']:+.3f}; paired by rat p vs baseline "
      f"{res_nb['paired_p_vs_baseline']:.3g}, vs non-target {res_nb['paired_p_vs_nontarget']:.3g}")

**Result: not reproduced.** The normalized fast ripple rate after target stimulation (+14.8) is
not below baseline (+8.5) or non-target (+4.8) in the main reading (paired by rat p 0.58 and
0.44); R3's target group (+109) dominates the target mean. The target mean is below both others
in none of the 12 readings, and no comparison is significant with target lower. By the median
per-window change the target is lowest (-0.19 against 0.00 baseline and +0.79 non-target), which
the criterion does not test. With the baseline seizures that carry trains left out, baseline is
+8.1 and the paired p values are 0.57 and 0.44.

## Summary

| Claim | Outcome |
|---|---|
| tables1-network-stable-short-term | reproduced (under the self-pairs reading; 5 of 7 rats without it; rests on the onset assumption) |
| fig2-r1-target-shifts | reproduced (rests on the onset assumption) |
| figs2-target-assignment | partially reproduced (18 of 28 baseline groups) |
| fig3a-target-reduces-each-rat | not reproduced (3 of 8 target groups below baseline) |
| fig3a-nontarget-can-lengthen | reproduced (13 of 20) |
| fig3a-per-rat-site-differences | partially reproduced (5 of 11 pairs unadjusted, 4 adjusted) |
| fig3a-target-vs-nontarget-total | not reproduced (-0.026 against 0.066, p = 0.16) |
| fig3a-some-nontarget-shorten | not reproduced |
| fig3b-same-site-target-vs-nontarget | not reproduced (ANT only) |
| seizure-duration-proportion | blocked: data missing |
| fig3c-frequency-no-difference | blocked: data missing |
| fig4b-psd-target-bands | not reproduced |
| fig4b-psd-other-bands-unchanged | not reproduced |
| fig5b-iis-post-seizure | reproduced (under the time-bin test; rests on the seizure end assumption) |
| fig5c-interictal-iis-no-difference | blocked: data missing |
| fig5d-iis-seizure-correlation | blocked: data missing |
| fig6cd-interictal-hfo-no-difference | blocked: data missing |
| fig6e-ictal-ripples-no-difference | reproduced (p = 0.061; 5 of 12 readings significant) |
| fig6f-ictal-fast-ripples-reduced | not reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The dandiset gives no seizure onset or end, so both
are found here from the LFP with a line-length rule of this attempt; every duration, GC, PSD, IIS
and HFO result rests on it, and thresholds 1.5 and 3 are shown beside the main 2. The stimulation
times are found from the stimulation artifact. GC is conditional time-domain GC from OLS VAR fits
written here, not the MVGC toolbox; the residuals at the BIC order are not white, and at a
fixed order of 40, where they are close to white, the three network outcomes hold as at the BIC
order. HFOs are
detected with the HFO App's short line length detector at its dialog defaults, read from its
source, without the paper's manual review; the epoch the threshold was computed over is assumed
to be the 600 s file epoch. The PSD and HFO post window ends where the next train starts. IIS
thresholds use the 180 s before each seizure, as the paper's interictal segments are not in the
dandiset. Target labels are those of Figure S2.

**What was found about the dandiset.**
- Each file holds 600 s epochs around seizures, with no seizure times, stimulation times,
  absolute start times or recording durations, and no interictal segments: five claims cannot be
  tested.
- The order of the stimulation groups differs from Figure S2 for R5 (CA1, ANT, DG, CA3 against
  ANT, DG, CA1, CA3) and R6 (ANT, DG, CA1, CA3 against CA1, ANT, DG, CA3); the epoch dates follow
  the dandiset's order.
- R3's CA1 stimulation group (R3-07) is dated 2023-10-18 to 10-20, within the dates of R3's DG
  group (R3-05, 10-18 to 10-22) and before R3's last baseline group (R3-06, 10-22 to 10-23),
  against the alternating order of the file names; its trains have R3-05's pulse rate (131.75 Hz),
  not that of R3's first two stimulation groups (125.25 and 125.75 Hz).
- Stimulation trains are found in every seizure of 24 of the 28 stimulation groups, but in 3 of 13,
  2 of 7 and 0 of 4 seizures of R7's ANT, CA1 and CA3 groups (and 7 of 12 of its DG group). Of the
  R7 seizures without a train, none in the CA1 and CA3 groups has a run of 2 s or more at R7's
  pulse rates; in the DG and ANT groups 3 of 5 and 3 of 10 have a run of 2 to 3.75 s inside the
  seizure, shorter than the 5 s trains of Methods C, which may be trains the 4 s rule misses.
- Seven baseline epochs carry 4.25 to 5.5 s trains at their rat's own pulse rate (R1-04 twice, R4-00
  three times, R5-00 and R6-02 once), so these baseline seizures appear to have been stimulated.
- Epoch dates of some groups overlap those of the next group (for example R1 2023-09-20 in both
  base3 and DG stimulation), and the date format varies (R4 uses `2023_9-11`). No epoch or channel
  is duplicated.
